# Statistical Inferences: Random Sampling and Estimator Expectations

In quantitative research, we rarely have access to the infinite data-generating process of a strategy or market (the **Population**). Instead, we must make critical risk allocations based on a localized window of historical data (the **Sample**).

---

## 1. The Front Desk Scenario: The Alpha Backtest

Imagine you are a Quant Research Analyst at a systematic hedge fund. Your team has engineered a new machine-learning algorithm that generates a stream of trade returns. You want to know the true underlying risk parameters of this strategy:
* What is its **True Expected Return ($\mu$)**?
* What is its **True Underlying Variance ($\sigma^2$)**?

Because you only have a finite sample of $n$ recorded historical trades, the statistics you compute ($\bar{X}$ and $s^2$) are *themselves* random variables. If you pull a different window of 100 trades tomorrow, your sample mean and sample variance will change. Our goal is to evaluate if these sample metrics are reliable, unbiased proxies for the hidden truth.

---

## 2. The Expected Value of the Sample Mean ($\bar{X}$)

To estimate the unknown population mean $\mu$, we calculate the standard arithmetic average of our observed sample:

$$\bar{X} = \frac{1}{n} \sum_{i=1}^n X_i$$

### Mathematical Proof of Unbiasedness
To verify if the sample mean is a reliable target, we take its mathematical expectation:

$$\mathbb{E}[\bar{X}] = \mathbb{E}\left[ \frac{1}{n} \sum_{i=1}^n X_i \right]$$

By the linearity property of expectations, we can pull the constant factor $\frac{1}{n}$ and the summation operator outside of the expectation wrapper:

$$\mathbb{E}[\bar{X}] = \frac{1}{n} \sum_{i=1}^n \mathbb{E}[X_i]$$

Since each individual trade realization $X_i$ is drawn directly from the underlying population, its individual expected value is exactly the population mean ($\mathbb{E}[X_i] = \mu$). Substituting this back in:

$$\mathbb{E}[\bar{X}] = \frac{1}{n} \sum_{i=1}^n \mu = \frac{1}{n} (n\mu) = \mu$$

* **Desk Takeaway:** Because $\mathbb{E}[\bar{X}] = \mu$, the sample mean is a perfectly **unbiased estimator** of the population mean.
* **The Sampling Uncertainty:** While unbiased, this estimator still possesses noise. Because the trades are independent, the variance of our estimator scales inversely with sample size:
$$\text{Var}(\bar{X}) = \text{Var}\left( \frac{1}{n} \sum_{i=1}^n X_i \right) = \frac{1}{n^2} \sum_{i=1}^n \text{Var}(X_i) = \frac{1}{n^2}(n\sigma^2) = \frac{\sigma^2}{n}$$
Taking the square root yields the **Standard Error ($\text{SE} = \frac{\sigma}{\sqrt{n}}$)**. To cut your estimation uncertainty in half, you must collect four times as much data.

---

## 3. The Expected Value of the Sample Variance ($s^2$) and Bessel's Correction

To estimate the hidden population variance $\sigma^2$, a naive approach would calculate the average squared deviation of our sample points from our sample mean:

$$\sigma^2_{\text{naive}} = \frac{1}{n} \sum_{i=1}^n (X_i - \bar{X})^2$$

### The Mathematical Trap: Why Naive Variance Underreports Risk
If you take the mathematical expectation of this naive estimator, it does *not* equal the population variance $\sigma^2$. Instead, it systematically underestimates it:

$$\mathbb{E}\left[ \sigma^2_{\text{naive}} \right] = \frac{n-1}{n}\sigma^2$$

Because the observed sample points $X_i$ are naturally closer to their *own sample average* ($\bar{X}$) than they are to the *true, hidden population mean* ($\mu$), the naive calculation shrinks the apparent variance by a factor of $\frac{n-1}{n}$.

### The Structural Proof
To see why this bias occurs, let's expand the squared deviation term by tricking the algebra—adding and subtracting the true population mean $\mu$ inside the parenthesis:

$$\sum_{i=1}^n (X_i - \bar{X})^2 = \sum_{i=1}^n \left[ (X_i - \mu) - (\bar{X} - \mu) \right]^2$$

Expanding the quadratic expression ($(a-b)^2 = a^2 - 2ab + b^2$) gives:

$$\sum_{i=1}^n (X_i - \bar{X})^2 = \sum_{i=1}^n (X_i - \mu)^2 - 2(\bar{X} - \mu)\sum_{i=1}^n (X_i - \mu) + \sum_{i=1}^n (\bar{X} - \mu)^2$$

Notice that $\sum_{i=1}^n (X_i - \mu) = n(\bar{X} - \mu)$. Substituting this into the cross-term simplifies the equation:

$$\sum_{i=1}^n (X_i - \bar{X})^2 = \sum_{i=1}^n (X_i - \mu)^2 - 2n(\bar{X} - \mu)^2 + n(\bar{X} - \mu)^2$$
$$\sum_{i=1}^n (X_i - \bar{X})^2 = \sum_{i=1}^n (X_i - \mu)^2 - n(\bar{X} - \mu)^2$$

Now, if we apply the expectation operator $\mathbb{E}[\cdot]$ to this identity:

$$\mathbb{E}\left[ \sum_{i=1}^n (X_i - \bar{X})^2 \right] = \sum_{i=1}^n \mathbb{E}[(X_i - \mu)^2] - n\mathbb{E}[(\bar{X} - \mu)^2]$$

By definition, $\mathbb{E}[(X_i - \mu)^2] = \sigma^2$ (population variance) and $\mathbb{E}[(\bar{X} - \mu)^2] = \text{Var}(\bar{X}) = \frac{\sigma^2}{n}$. Plugging these parameters in:

$$\mathbb{E}\left[ \sum_{i=1}^n (X_i - \bar{X})^2 \right] = n\sigma^2 - n\left(\frac{\sigma^2}{n}\right) = n\sigma^2 - \sigma^2 = (n-1)\sigma^2$$

### The Solution: Bessel's Correction ($n-1$)
To completely eliminate this downward bias, we multiply the equation by $\frac{1}{n-1}$. This defines the standard **Sample Variance ($s^2$)**:

$$s^2 = \frac{1}{n-1} \sum_{i=1}^n (X_i - \bar{X})^2$$

Taking its expectation now maps perfectly onto the target:

$$\mathbb{E}[s^2] = \mathbb{E}\left[ \frac{1}{n-1} \sum_{i=1}^n (X_i - \bar{X})^2 \right] = \frac{1}{n-1} \left( (n-1)\sigma^2 \right) = \sigma^2$$

* **Desk Takeaway:** When calculating risk margins or standard deviations from historical sample trade logs, always enforce Bessel's Correction by dividing by $n-1$ (set `ddof=1` in Python). If you divide by $n$, you are underreporting the actual volatility of your trading asset.

---

## 4. Operational Summary Matrix

| Metric Name | Mathematical Definition | Expectation $\mathbb{E}[\cdot]$ | Bias Profile | Quantitative Function |
| :--- | :--- | :--- | :--- | :--- |
| **Sample Mean ($\bar{X}$)** | $\frac{1}{n}\sum X_i$ | $\mu$ | **Unbiased** | Tracks strategy's central return capability |
| **Naive Variance ($\sigma^2_{\text{naive}}$)** | $\frac{1}{n}\sum(X_i - \bar{X})^2$ | $\frac{n-1}{n}\sigma^2$ | **Biased (Downward)** | Deficient metric; underreports true financial risk scale |
| **Sample Variance ($s^2$)** | $\frac{1}{n-1}\sum(X_i - \bar{X})^2$ | $\sigma^2$ | **Unbiased** | Core variance target used to define active portfolio margin controls |

In [ ]:
import numpy as np

def run_sampling_expectation_demo():
    # 1. Define the true underlying Population parameters (The absolute hidden truth)
    # Let's say our trading strategy has a true mean return of 2.0% and a true variance of 9.0% (Vol = 3.0%)
    true_mu = 2.0
    true_sigma2 = 9.0
    true_vol = np.sqrt(true_sigma2)

    # 2. Sampling parameters
    sample_size = 10  # Keeping sample size small to heavily expose the Bessel bias!
    n_simulations = 50000 # Pulling 50,000 independent samples to observe the expected values

    sample_means = []
    naive_variances = []
    corrected_variances = []

    # 3. Execute Monte Carlo Sampling Engine
    np.random.seed(42)
    for _ in range(n_simulations):
        # Sample n trades randomly from our population distribution
        sample = np.random.normal(loc=true_mu, scale=true_vol, size=sample_size)

        # Calculate sample mean
        x_bar = np.mean(sample)
        sample_means.append(x_bar)

        # Calculate Naive Variance (Divided by n, ddof=0)
        naive_var = np.var(sample, ddof=0)
        naive_variances.append(naive_var)

        # Calculate Corrected Sample Variance (Divided by n-1, ddof=1 via Bessel's Correction)
        corrected_var = np.var(sample, ddof=1)
        corrected_variances.append(corrected_var)

    print("=========================================================")
    print("         SAMPLING RISK & ESTIMATOR EXPECTATIONS          ")
    print("=========================================================")
    print(f"True Population Mean (μ)         : {true_mu:.4f}%")
    print(f"True Population Variance (σ²)     : {true_sigma2:.4f}")
    print(f"Sample Size (n)                  : {sample_size} trades per sample")
    print("---------------------------------------------------------")
    print(f"Expected Value of Sample Mean E[X̄] : {np.mean(sample_means):.4f}%  <-- Perfectly Unbiased!")
    print("---------------------------------------------------------")
    print(f"Expected Value of Naive Var (n)  : {np.mean(naive_variances):.4f}   <-- Underreports risk! Bias = {(sample_size-1)/sample_size:.2f}*σ²")
    print(f"Expected Value of Sample Var (n-1): {np.mean(corrected_variances):.4f}   <-- Perfectly Unbiased Risk Target!")
    print("=========================================================")

run_sampling_expectation_demo()

## 5. Visualizing the Sampling Distribution

To truly grasp the relationship between a **Population** and a **Sample**, we must observe them simultaneously.

The interactive dashboard below runs a live sampling engine. On the **Left Panel**, you see the true underlying population distribution alongside an individual empirical sample of size $n$. On the **Right Panel**, you see the **Sampling Distribution of the Sample Mean**—the distribution of thousands of sample means calculated from independent runs.

> 🎛️ **Interactive Exercise:** Use the slider at the bottom of the dashboard to increase the sample size $n$. Notice how increasing $n$ does **not** change the shape of the underlying population on the left, but it forces the distribution of the sample mean on the right to collapse tightly onto the true population mean ($\mu$). This is visual proof that $\text{Var}(\bar{X}) = \frac{\sigma^2}{n}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Setup the Fixed Population Parameters (The Absolute Truth)
true_mu = 2.0
true_sigma = 3.0
np.random.seed(42)

# Pre-generate a massive population array to simulate the "infinite" process
population_data = np.random.normal(loc=true_mu, scale=true_sigma, size=100000)

# Pre-generate 5,000 independent sample runs up to max size n=100 for the right plot
n_simulations = 5000
max_n = 100
all_simulated_samples = np.random.normal(loc=true_mu, scale=true_sigma, size=(n_simulations, max_n))

# 2. Define the Live Rendering Function
def render_sampling_dashboard(n):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f"The Sampling Engine Dashboard (At Sample Size n = {n} Observations)", fontsize=16, fontweight='bold')

    # --- LEFT GRAPH: Population vs. A Single Realized Sample ---
    # Plot true population background
    ax1.hist(population_data, bins=100, density=True, color='gray', alpha=0.15, label='Infinite Population Background')

    # Pull out ONE specific single sample slice of size n to show what a quant actually sees
    single_sample = all_simulated_samples[0, :n]

    # Plot the empirical histogram of that single sample
    ax1.hist(single_sample, bins=max(5, n//4), density=True, color='tab:blue', alpha=0.5,
             edgecolor='white', label=f'A Single Sample (n={n})')

    # Highlight the single sample mean vs true mean
    current_sample_mean = np.mean(single_sample)
    ax1.axvline(true_mu, color='red', linestyle='-', linewidth=2, label=f'True Mean μ ({true_mu:.1f}%)')
    ax1.axvline(current_sample_mean, color='tab:blue', linestyle='--', linewidth=2.5,
                label=f'Sample Mean X̄ ({current_sample_mean:.2f}%)')

    ax1.set_xlim(-10, 14)
    ax1.set_ylim(0, 0.45)
    ax1.set_title("The Ground Reality\nStatic Population vs. Individual Sample Spreadsheet")
    ax1.set_xlabel("Strategy Return (%)")
    ax1.set_ylabel("Probability Density")
    ax1.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)

    # --- RIGHT GRAPH: The Sampling Distribution of the Sample Mean ---
    # Calculate the mean of ALL 5,000 independent simulation universes up to size n
    means_distribution = np.mean(all_simulated_samples[:, :n], axis=1)

    # Plot the distribution of those means
    ax2.hist(means_distribution, bins=40, density=True, color='teal', alpha=0.6, edgecolor='white',
             label='Distribution of X̄ over 5,000 Retrials')

    # Add a reference line for the true target mean
    ax2.axvline(true_mu, color='red', linestyle='-', linewidth=2)

    # Overlay the theoretical standard error curve predicted by Var(X̄) = sigma^2 / n
    standard_error = true_sigma / np.sqrt(n)
    x_axis = np.linspace(-10, 14, 500)
    from scipy.stats import norm
    theoretical_se_curve = norm.pdf(x_axis, loc=true_mu, scale=standard_error)
    ax2.plot(x_axis, theoretical_se_curve, color='crimson', lw=2.5, linestyle='--', label=f'Theoretical SE Envelope')

    ax2.set_xlim(-5, 9)
    ax2.set_ylim(0, 1.5)  # Let it scale high so you see it compress!
    ax2.set_title("The Estimator Meta-Universe\nSampling Distribution of the Sample Mean (X̄)")
    ax2.set_xlabel("Calculated Value of Sample Mean (X̄)")
    ax2.set_ylabel("Probability Density")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 3. Handle Widget Placements
output_area = widgets.Output()

slider = widgets.IntSlider(
    value=5,
    min=2,
    max=100,
    step=1,
    description='Sample Size (n):',
    continuous_update=True,
    layout=widgets.Layout(width='65%', margin='10px 0px 0px 50px')
)

def on_slider_change(change):
    with output_area:
        clear_output(wait=True)
        render_dashboard = render_sampling_dashboard(change['new'])

slider.observe(on_slider_change, names='value')

# Seed the initial plot layout at n=5
with output_area:
    render_sampling_dashboard(slider.value)

dashboard_layout = widgets.VBox([output_area, slider])
display(dashboard_layout)